In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/francescomanzoni/vulnerability-management-datasets/cve_corpus.csv
/kaggle/input/datasets/francescomanzoni/vulnerability-management-datasets/cve_cisa_epss_enriched_dataset.csv


In [2]:
# ---------------------------------------------------
# Vulcan CVE Attack Complexity Prediction
# ---------------------------------------------------
# Author: [Your Name]
# Dataset: CVE_CISA_EPSS_enriched
# Goal: Predict attack_complexity based on CVE features
# ---------------------------------------------------

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# ---------------------------------------------------
# 1) Load dataset
# ---------------------------------------------------
file_path = "/kaggle/input/datasets/francescomanzoni/vulnerability-management-datasets/cve_cisa_epss_enriched_dataset.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # Clean column names

# ---------------------------------------------------
# 2) Feature engineering
#    a) Vulnerability type (categorical)
# ---------------------------------------------------
def get_vulnerability_type(desc):
    if pd.isna(desc):
        return "Unknown"
    desc = str(desc).lower()
    if "sql injection" in desc or "sqli" in desc:
        return "SQL Injection"
    elif "cross-site scripting" in desc or "xss" in desc:
        return "Cross-Site Scripting (XSS)"
    elif "buffer overflow" in desc:
        return "Buffer Overflow"
    elif "denial of service" in desc or "dos" in desc:
        return "Denial of Service (DoS)"
    elif "privilege escalation" in desc:
        return "Privilege Escalation"
    elif "remote code execution" in desc or "rce" in desc:
        return "Remote Code Execution (RCE)"
    elif "directory traversal" in desc or "path traversal" in desc:
        return "Directory Traversal"
    elif "bypass" in desc:
        return "Security Bypass"
    elif "information disclosure" in desc or "leak" in desc:
        return "Information Disclosure"
    elif "cross-site request forgery" in desc or "csrf" in desc:
        return "CSRF"
    else:
        return "Other / Generic"

if "description" in df.columns:
    df["vulnerability"] = df["description"].apply(get_vulnerability_type)
else:
    df["vulnerability"] = "Unknown"

le_vuln = LabelEncoder()
df["vulnerability_encoded"] = le_vuln.fit_transform(df["vulnerability"].astype(str))

# ---------------------------------------------------
# 3) Attack vector (categorical)
# ---------------------------------------------------
if "attack_vector" not in df.columns:
    df["attack_vector"] = "Unknown"  # Safe default

le_vector = LabelEncoder()
df["attack_vector_encoded"] = le_vector.fit_transform(df["attack_vector"].astype(str))

# ---------------------------------------------------
# 4) Asset Criticality (deterministic mapping)
# ---------------------------------------------------
severity_map = {"CRITICAL": 10, "HIGH": 8, "MEDIUM": 6, "LOW": 3}
if "base_severity" in df.columns:
    df["Asset_Criticality"] = (
        df["base_severity"].astype(str).str.upper().map(severity_map).fillna(5).astype(int)
    )
else:
    df["Asset_Criticality"] = 5

# ---------------------------------------------------
# 5) AI Risk Score (for analysis only)
# ---------------------------------------------------
if "base_score" in df.columns:
    df["AI_Risk_Score"] = ((df["base_score"] * 0.6 + df["Asset_Criticality"] * 0.4) * 10).round(1)
else:
    df["AI_Risk_Score"] = df["Asset_Criticality"] * 10

# ---------------------------------------------------
# 6) Define features and target
# ---------------------------------------------------
features = [
    "base_score",
    "exploitability_score",
    "impact_score",
    "epss_score",
    "epss_perc",
    "vulnerability_encoded",
    "attack_vector_encoded"
]

# Keep only existing columns
features = [col for col in features if col in df.columns]

if "attack_complexity_encoded" in df.columns:
    y = df["attack_complexity_encoded"].copy()
    le_target = None
elif "attack_complexity" in df.columns:
    le_target = LabelEncoder()
    y = le_target.fit_transform(df["attack_complexity"].astype(str))
else:
    raise KeyError("Target column 'attack_complexity' not found.")

X = df[features].copy()
print("X shape:", X.shape)
print("y shape:", y.shape)

# ---------------------------------------------------
# 7) Train/test split (stratified)
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------------------------------
# 8) Random Forest classifier with hyperparameter tuning
# ---------------------------------------------------
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 8, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "criterion": ["gini", "entropy"]
}

rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(rf, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
print("\nBest Hyperparameters:")
print(grid.best_params_)

# ---------------------------------------------------
# 9) Evaluation
# ---------------------------------------------------
y_pred_train = best_rf.predict(X_train)
y_pred_test = best_rf.predict(X_test)

print(f"\nTraining Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Macro F1-score: {f1_score(y_test, y_pred_test, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

# ---------------------------------------------------
# 10) Save model and encoders
# ---------------------------------------------------
pipeline_data = {
    "model": best_rf,
    "le_vuln": le_vuln,
    "le_vector": le_vector,
    "expected_features": features
}
if le_target:
    pipeline_data["le_target"] = le_target

joblib.dump(pipeline_data, "Vulcan_RF_Pipeline_Final.pkl")
print("\nImproved model saved successfully 🚀")

X shape: (155852, 7)
y shape: (155852,)
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Hyperparameters:
{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

Training Accuracy: 1.0000
Test Accuracy: 0.9988
Macro F1-score: 0.9933

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1493
           1       1.00      1.00      1.00     29678

    accuracy                           1.00     31171
   macro avg       1.00      0.99      0.99     31171
weighted avg       1.00      1.00      1.00     31171

Confusion Matrix:
[[ 1465    28]
 [   10 29668]]

Improved model saved successfully 🚀
